In [1]:
# Installing yellowbrick
!pip install yellowbrick

In [2]:
# optional formatter setup; do not fail the notebook if nb_black is unavailable
try:
    %load_ext nb_black
except Exception:
    pass

import warnings

warnings.filterwarnings("ignore")


# Libraries to help with reading and manipulating data
import numpy as np
import pandas as pd

# to perform PCA
from sklearn.decomposition import PCA

# Libraries to help with data visualization
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()

# Removes the limit for the number of displayed columns
pd.set_option("display.max_columns", None)
# Sets the limit for the number of displayed rows
pd.set_option("display.max_rows", 200)

# to scale the data using z-score
from sklearn.preprocessing import StandardScaler

# to compute distances
from scipy.spatial.distance import cdist

# to perform k-means clustering and compute silhouette scores
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# to visualize the elbow curve and silhouette scores
from yellowbrick.cluster import KElbowVisualizer, SilhouetteVisualizer

In [3]:
# Loading dataset
assessment_df =pd.read_csv('assessments.csv')

In [4]:
assessment_df.shape

(206, 6)

# assessmet data set having 206 rows and 6 columns

In [5]:
assessment_df.head()

,code_module,code_presentation,id_assessment,assessment_type,date,weight
0,AAA,2013J,1752,TMA,19.0,10.0
1,AAA,2013J,1753,TMA,54.0,20.0
2,AAA,2013J,1754,TMA,117.0,20.0
3,AAA,2013J,1755,TMA,166.0,20.0
4,AAA,2013J,1756,TMA,215.0,30.0


In [6]:
assessment_df.tail()

,code_module,code_presentation,id_assessment,assessment_type,date,weight
201,GGG,2014J,37443,CMA,229.0,0.0
202,GGG,2014J,37435,TMA,61.0,0.0
203,GGG,2014J,37436,TMA,124.0,0.0
204,GGG,2014J,37437,TMA,173.0,0.0
205,GGG,2014J,37444,Exam,229.0,100.0


In [7]:
assessment_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 206 entries, 0 to 205
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   code_module        206 non-null    object 
 1   code_presentation  206 non-null    object 
 2   id_assessment      206 non-null    int64  
 3   assessment_type    206 non-null    object 
 4   date               195 non-null    float64
 5   weight             206 non-null    float64
dtypes: float64(2), int64(1), object(3)
memory usage: 9.8+ KB


In [8]:
assessment_df.isnull().sum()

code_module           0
code_presentation     0
id_assessment         0
assessment_type       0
date                 11
weight                0
dtype: int64

In [9]:
assessment_df.describe().T

,count,mean,std,min,25%,50%,75%,max
id_assessment,206.0,26473.975728,10098.625521,1752.0,15023.25,25364.5,34891.75,40088.0
date,195.0,145.005128,76.001119,12.0,71.00,152.0,222.00,261.0
weight,206.0,20.873786,30.384224,0.0,0.00,12.5,24.25,100.0


In [10]:
#checking duplicated values
assessment_df[assessment_df.duplicated(keep=False)]

,code_module,code_presentation,id_assessment,assessment_type,date,weight


# assessment data set did not have duplicated values.
# have 11 missing values in date column
# fill with mean value to avoid outliers

In [11]:
# Very few values were missing 11 out of 206 rows,
# considered as small dataset and filled with mean value of date column to avoid outliers.
assessment_df = assessment_df.fillna(145)

In [12]:
assessment_df.isnull().sum()

code_module          0
code_presentation    0
id_assessment        0
assessment_type      0
date                 0
weight               0
dtype: int64

In [13]:
# data cleaning with lower case to use model fit in one format only.
for col in assessment_df.select_dtypes(include='object',).columns:
   assessment_df[col] = assessment_df[col].str.strip().str.lower()

In [14]:
#checking dataset format change from upper case to lower case
# data cleaning with lower case to use model fit in one format only.
for col in assessment_df.select_dtypes(include='object',).columns:
    print(col)
    print(assessment_df[col].unique())

code_module
['aaa' 'bbb' 'ccc' 'ddd' 'eee' 'fff' 'ggg']
code_presentation
['2013j' '2014j' '2013b' '2014b']
assessment_type
['tma' 'exam' 'cma']


In [15]:
#data merging 
#Merging dataset with columns of code_module and code_presentation
assessment_df["code_module_presentation"]=assessment_df[['code_module','code_presentation']].agg('-'.join,axis=1)
assessment_df = assessment_df.drop(columns=['code_module','code_presentation'])
assessment_df.head()

,id_assessment,assessment_type,date,weight,code_module_presentation
0,1752,tma,19.0,10.0,aaa-2013j
1,1753,tma,54.0,20.0,aaa-2013j
2,1754,tma,117.0,20.0,aaa-2013j
3,1755,tma,166.0,20.0,aaa-2013j
4,1756,tma,215.0,30.0,aaa-2013j


In [16]:
#create newdataset with new name as after merging and use for EDA
assessment_df1 = assessment_df

In [17]:
assessment_df1.head

<bound method NDFrame.head of      id_assessment assessment_type   date  weight code_module_presentation
0             1752             tma   19.0    10.0                aaa-2013j
1             1753             tma   54.0    20.0                aaa-2013j
2             1754             tma  117.0    20.0                aaa-2013j
3             1755             tma  166.0    20.0                aaa-2013j
4             1756             tma  215.0    30.0                aaa-2013j
..             ...             ...    ...     ...                      ...
201          37443             cma  229.0     0.0                ggg-2014j
202          37435             tma   61.0     0.0                ggg-2014j
203          37436             tma  124.0     0.0                ggg-2014j
204          37437             tma  173.0     0.0                ggg-2014j
205          37444            exam  229.0   100.0                ggg-2014j

[206 rows x 5 columns]>

# assessment dataset having information about assessment type ,date of assessment and weightage of assessment results.
# thease data is not required for 25% early student drop out prediction.
# this data is required when exams were conducted before joining course for adimission which were students fail .
# one more scenario is like that once completed one chapter and conducted exams to improve student skills.
these aabove said cases ,we can consider these datasets to feed ML model.